# Module 20 — Checkpointing, interrupts and time travel

**THE ONE IDEA:** this is the **only** reason to reach for LangGraph. Module 19 showed the
graph adds no capability over a `while` loop. These three do:

| | what it is | hand-rolling it |
|---|---|---|
| **`thread_id`** | state persisted per conversation | you write a store and a schema |
| **`interrupt()`** | pause mid-run, resume later, in a different process | you write a resumable state machine |
| **time travel** | fork history at any past step and re-run | you version every intermediate state |

Module 14 built the human gate by hand in ~40 lines. Here it is one function call — but
notice that module 14's gate could not **survive a process restart**. This one can.

Runs on `_fake_model`: free, deterministic, no API key.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from _fake_model import FakeModel, tool_turn, text_turn
from _tools import run_tool, WRITE_TOOLS

class S(TypedDict):
    messages: Annotated[list, operator.add]
    audit:    Annotated[list, operator.add]

SCRIPT = [tool_turn("fetch_customer_note", {"customer_id": "C-1002"}, "a"),
          tool_turn("confirm_decision", {"reference": "ERC-WAIVER-1002"}, "b"),
          text_turn("Waiver recorded.")]


## The graph — one node contains the gate

In [ ]:
def agent(state: S):
    # The script position is DERIVED FROM STATE, not from a global counter.
    # A FakeModel holding its own `self.calls` would break time travel: replaying
    # an old checkpoint would resume mid-script instead of re-deriving. Anything
    # stateful OUTSIDE the graph state does not time-travel. That is a real trap.
    turn = sum(1 for a in state["audit"] if a.startswith("llm:"))
    r = FakeModel(SCRIPT[turn:]).create(messages=state["messages"])
    return {"messages": [r.choices[0].message], "audit": [f"llm:{r.choices[0].finish_reason}"]}

def tools(state: S):
    out, audit = [], []
    for tc in state["messages"][-1].tool_calls:
        name, args = tc.function.name, json.loads(tc.function.arguments)

        if name in WRITE_TOOLS:
            # PAUSES THE GRAPH. State is checkpointed; the process may now exit.
            decision = interrupt({"tool": name, "args": args,
                                  "question": f"Approve {name}({args})?"})
            audit.append(f"gate:{name}:{decision}")
            if decision != "approve":
                out.append({"role": "user", "content": "DENIED by a human."})
                continue
        out.append({"role": "user", "content": run_tool(name, args)})
        audit.append(f"ran:{name}")
    return {"messages": out, "audit": audit}

def route(state: S):
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else END

b = StateGraph(S)
b.add_node("agent", agent); b.add_node("tools", tools)
b.add_edge(START, "agent")
b.add_conditional_edges("agent", route, {"tools": "tools", END: END})
b.add_edge("tools", "agent")
graph = b.compile(checkpointer=MemorySaver())      # <- HITL REQUIRES a checkpointer
cfg = {"configurable": {"thread_id": "case-1002"}}
print("compiled with MemorySaver; thread_id =", cfg["configurable"]["thread_id"])

## Run until it hits the gate

In [ ]:
r = graph.invoke({"messages": [{"role": "user", "content": "Approve the ERC waiver for C-1002."}],
                  "audit": []}, cfg)

print("paused:", "__interrupt__" in r)
print("payload:", r["__interrupt__"][0].value)
print("graph is parked at node:", graph.get_state(cfg).next)
print("\n^ the process could exit here. State is in the checkpointer, keyed by")
print("  thread_id. Module 14's in-memory gate could not survive that.")

## Resume with the human's answer

In [ ]:
final = graph.invoke(Command(resume="approve"), cfg)
print("audit trail:", final["audit"])
print("answer     :", final["messages"][-1].content)

## Time travel — re-run the same decision point, denied

Fork history at the checkpoint *before* the gate and take the other branch. You cannot do
this without per-step persistence.

In [ ]:
history = list(graph.get_state_history(cfg))
print(f"{len(history)} checkpoints — every step is replayable for INSPECTION:")
for h in reversed(history):
    n = len(h.values.get("messages", []))
    print(f"  next={str(h.next):12} messages={n:2}  audit={h.values.get('audit', [])[-1:]}")

# Take the OTHER branch. A fresh thread_id, because re-resuming an interrupt that
# has already been answered replays the RECORDED answer — see the trap below.
cfg2 = {"configurable": {"thread_id": "case-1002-denied"}}
graph.invoke({"messages": [{"role": "user", "content": "Approve the ERC waiver for C-1002."}],
              "audit": []}, cfg2)
denied = graph.invoke(Command(resume="deny"), cfg2)

print("\napproved thread audit:", final["audit"])
print("denied   thread audit:", denied["audit"])
committed = lambda st: any("COMMITTED" in str(m.get("content", ""))
                           for m in st["messages"] if isinstance(m, dict))
print(f"\ncommitted? approved={committed(final)}  denied={committed(denied)}")

print("""
LESSON - three things, and only these three, justify the framework:

  thread_id    state lives outside the process. Restart the worker, resume the
               case. Swap MemorySaver for PostgresSaver and any worker picks up
               any thread - that is how HITL works at scale.

  interrupt()  the pause is a first-class state, not a blocked thread. The human
               may answer in 3 seconds or 3 days, from a different machine.

  checkpoints  every step is recorded, so you can inspect exactly what the state
               was and what was pending at any point - the table above.

Module 14 built this gate by hand and it worked, but it held state in a Python
dict: a restart lost the case. THAT is the gap LangGraph fills, and it is the
only honest argument for adding it.

TWO TRAPS, both found by running this notebook rather than reading the docs:

  1. Re-invoking Command(resume=...) against an OLD checkpoint on the SAME thread
     REPLAYS THE RECORDED ANSWER. It does not fork. The first draft "forked" to
     deny and got approve back, with no error. To take a different branch, use a
     different thread_id - as above.

  2. The agent node derives its script position from STATE. An earlier draft used
     a module-level FakeModel with its own counter; replaying a checkpoint then
     resumed mid-script and produced a plausible but meaningless result. Anything
     stateful OUTSIDE graph state does not replay correctly.

You will also see msgpack warnings about _fake_model types - the checkpointer is
serialising dataclasses it does not know. Harmless here; in production you
checkpoint plain dicts.""")

---

**Next:** `21_langgraph_parallel_subgraphs_streaming.ipynb`